# Project 3 — Fraud Detection: Build & Serve

**Duration:** 5 min

## Handle Imbalance & Train

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_curve
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib

df = pd.read_csv('creditcard.csv')
print(df['Class'].value_counts())  # 0: 284315, 1: 492

X = df.drop(columns=['Class','Time'])
y = df['Class']

scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)
# SMOTE on train only
X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f'After SMOTE: {y_train_res.value_counts().to_dict()}')

model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                           scale_pos_weight=1, eval_metric='aucpr', random_state=42)
model.fit(X_train_res, y_train_res)

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-advanced/ap-project3-build.ipynb)


## Threshold Tuning

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

# Find threshold maximising F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f'Best threshold: {best_threshold:.3f}, F1: {f1_scores[best_idx]:.3f}')

preds = (probs >= best_threshold).astype(int)
print(classification_report(y_test, preds, target_names=['legit','fraud']))

joblib.dump({'model': model, 'scaler': scaler, 'threshold': best_threshold}, 'fraud_model.pkl')

## FastAPI Serving Endpoint

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np

app = FastAPI()
bundle = joblib.load('fraud_model.pkl')

class Transaction(BaseModel):
    features: list[float]  # V1-V28 + Amount (29 values)

@app.post('/predict')
def predict(tx: Transaction):
    X = np.array(tx.features).reshape(1, -1)
    X[:, -1] = bundle['scaler'].transform(X[:, -1].reshape(-1, 1)).flatten()
    prob = bundle['model'].predict_proba(X)[0, 1]
    return {
        'fraud': bool(prob >= bundle['threshold']),
        'probability': round(float(prob), 4)
    }

# Run: uvicorn serve:app --reload

```
POST /predict
{"features": [0.1, -1.3, 0.9, ...]}
→ {"fraud": false, "probability": 0.0023}
```

> **💡 Tip:** ✅ Project 3 complete! You've built a production fraud detection system with a REST API. Claim your certificate!